# Campaign example

This example reconstructs the measured FE–rate frontier from the final campaign data and demonstrates random initialization, boundary-repulsive farthest-point sampling (BR-FPS), and Bayesian optimization (BO).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import display

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "styles" / "manuscript.mplstyle").is_file()
     and (path / "src" / "campaign.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Start Jupyter in the repository or its notebooks folder.")

sys.path.insert(0, str(root))
from src.campaign import (
    BOSettings, FEATURE_COLUMNS, TARGET_COLUMNS, generate_initial_design,
    propose_bo_batch, select_space_filling,
)
from src.preprocessing import CONDITION
from src.pareto import campaign_objectives, observed_front
from src.preprocessing import load_campaign

style = root / "styles" / "manuscript.mplstyle"
use_tex = False
campaign = load_campaign(root)
display(campaign.history.groupby(["Search-space stage", "Selection strategy"]).size()
        .rename("Conditions").reset_index())
print(f"{len(campaign.eligible)} successful electrolyses")
print(f"{len(campaign.retained)} retained after IQR filtering")

## Experimental FE–rate frontier

Average the retained independent electrolyses within each condition. A condition is nondominated if no other measured condition has at least as high an FE and rate and exceeds it in at least one objective. Membership here is based on the condition means.

In [ ]:
conditions = observed_front(campaign_objectives(campaign.retained))
fe = "FE mean (%)"
rate = "Rate mean (nmol/s/cm^2)"
front = conditions.loc[conditions["Mean-based Pareto"]].sort_values(fe)
display(front[["Exp #", "n independent electrolysis", fe, rate]])

with plt.style.context(style):
    plt.rcParams["text.usetex"] = use_tex
    fig, ax = plt.subplots(figsize=(3.25, 3.25), layout="constrained")
    ax.scatter(conditions[fe], conditions[rate], color="0.7",
               label="Measured condition means")
    ax.plot(front[fe], front[rate], "o-", label="Nondominated means")
    for label, x, y in front[["Exp #", fe, rate]].itertuples(index=False, name=None):
        ax.annotate(label, (x, y), xytext=(3, 4), textcoords="offset points")
    ax.set(xlabel="Ammonia Faradaic efficiency (%)",
           ylabel=r"Ammonia rate (nmol s$^{-1}$ cm$^{-2}$)")
    ax.set_box_aspect(1)
    ax.legend(frameon=False)
    plt.show()

## Seeded initial-design example

Generate 64 candidate rows using the campaign sampler with explicit NumPy and Torch seeds. These rows demonstrate the sampler. Use the campaign history and replicate table for the actual measured conditions.

In [ ]:
initial_design = generate_initial_design(64, numpy_seed=42, torch_seed=12345)
display(initial_design.head())

## Propose conditions with BR-FPS

Use the first 16 measured conditions as the existing design. BR-FPS selects candidates that improve coverage while penalizing proximity to the bounds. It uses input settings alone. This example proposes four candidates from a reduced candidate pool for a quick demonstration.

In [ ]:
existing = campaign.condition_means.loc[
    campaign.condition_means[CONDITION].between(1, 16), list(FEATURE_COLUMNS)
]
fps_candidates = select_space_filling(
    existing, stage="initial", numpy_seed=123, torch_seed=12345,
    n_new=4, n_candidates=1000,
)
display(fps_candidates)

## Propose conditions with Bayesian optimization

Use the retained individual electrolysis rows for the same 16 conditions. The campaign BO method fits separate Gaussian processes for measured FE and rate. It uses constrained qLogNEHVI to propose a batch jointly. The example uses two candidates and reduced optimization settings for a shorter runtime. The returned acquisition value scores the proposed batch, and FE and rate would be measured after electrolysis.

In [ ]:
training = campaign.model_rows.loc[
    campaign.model_rows[CONDITION].between(1, 16),
    [*FEATURE_COLUMNS, *TARGET_COLUMNS],
]
settings = BOSettings(
    stage="initial", seed=12345, batch_size=2, mc_samples=16,
    num_restarts=2, raw_samples=64, n_burnin=100, n_thinning=4,
    batch_limit=2, maxiter=50,
)
bo_candidates, log_acquisition = propose_bo_batch(training, settings)
display(bo_candidates)
print(f"Batch log acquisition value: {log_acquisition:.6f}")